In [21]:
from typing import Annotated, TypedDict

from langchain.tools import tool
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_ollama.chat_models import ChatOllama
from langgraph.graph import StateGraph, add_messages


In [22]:
# -- 1) Define our single business tool
@tool
def cancel_order(order_id: str) -> str:
    """Cancel an order that hasn't shipped."""
    # (Here you'd call your real backend API)
    return f"Order {order_id} has been cancelled."
    

In [23]:
# -- 2) The agent "brain": invoke LLM, run tool, then invoke LLM again
def call_model(state):
    msgs = state["messages"]
    order = state.get("order", {"order_id": "UNKNOWN"})
    
    # System prompt tells the model exactly what to do
    prompt = (
        f'''You are an ecommerce support agent.
        ORDER ID: {order['order_id']}
        If the customer asks to cancel, call cancel_order(order_id) 
        and then send a simple confirmation.
        Otherwise, just respond normally.'''
        )
    
    full = [SystemMessage(prompt)] + msgs
    print(full)
    
    model = ChatOllama(model="qwen3:8b", temperature=0)
    model_with_tools = model.bind_tools([cancel_order])
    
     # First pass: ask the model whether to call the tool.
    first = model_with_tools.invoke(full)
    out = [first]
    
    if first.tool_calls:
        # Run the requested tool and give its result back to the model.
        tc = first.tool_calls[0]
        tool_result = cancel_order.invoke(tc["args"])
        out.append(
            ToolMessage(content=str(tool_result), tool_call_id=tc["id"])
        )

        # Second pass: write the customer-facing reply.
        second = model.invoke(full + out)
        out.append(second)

    return {"messages": out}

In [24]:
class OrderState(TypedDict):
    order: dict | None
    messages: Annotated[list[BaseMessage], add_messages]

In [25]:
# -- 3) Wire it all up in a StateGraph
def construct_graph():
    g = StateGraph(OrderState)
    g.add_node("assistant", call_model)
    g.set_entry_point("assistant")
    
    return g.compile()

In [26]:
graph = construct_graph()

In [27]:
example_order = {"order_id": "A12345"}
convo = [HumanMessage(content="Please cancel my order A12345.")]

result = graph.invoke({"order": example_order, "messages": convo})

for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

[SystemMessage(content='You are an ecommerce support agent.\n        ORDER ID: A12345\n        If the customer asks to cancel, call cancel_order(order_id) \n        and then send a simple confirmation.\n        Otherwise, just respond normally.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Please cancel my order A12345.', additional_kwargs={}, response_metadata={}, id='21035f32-3d3f-44f9-b566-e491ace6f399')]
human: Please cancel my order A12345.
ai: 
tool: Order A12345 has been cancelled.
ai: Your order A12345 has been successfully cancelled. If you need any further assistance, feel free to ask!


In [ ]:
41